# 03.03 — Data Profiling y calidad de datos

## Objetivo

En este notebook se realiza un perfil sistemático de las dos fuentes ingeridas en la etapa anterior: **NYC Yellow Taxi** y **NOAA GHCND**.

El objetivo no es limpiar ni modificar los datos, sino identificar su estructura, cobertura y principales problemas de calidad antes de definir las reglas de transformación del pipeline.

El análisis se concentra en:

1. volumen y esquema;
2. valores nulos y patrones de ausencia;
3. cardinalidad de variables categóricas;
4. rangos y distribución de variables numéricas;
5. consistencia temporal;
6. reglas lógicas de calidad;
7. cobertura de los datos meteorológicos.

Dado que la base de Taxi contiene más de **41 millones de registros**, el profiling se diseña para minimizar recorridos sobre los archivos Parquet y evitar persistir el dataset completo en memoria.


## 1. Configuración del entorno

Se cargan las dependencias necesarias y se crea la sesión de Spark utilizada durante el profiling.


In [55]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [56]:
spark = (
    SparkSession.builder
    .appName("03_03_data_profiling")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [57]:
print("=" * 60)
print("INFORMACIÓN DEL ENTORNO SPARK")
print("=" * 60)

print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

INFORMACIÓN DEL ENTORNO SPARK
Spark version: 4.1.3
Application name: 03_03_data_profiling
Master: local[*]
Default parallelism: 12


## 2. Fuentes de datos y carga

Se utilizan directamente los archivos almacenados en la capa `raw`. Esta etapa es únicamente diagnóstica: **ningún archivo original será modificado**.

- **NYC Yellow Taxi:** archivos Parquet mensuales de 2024.
- **NOAA:** respuestas JSON de la estación de Central Park para 2024.


In [58]:
YEAR = 2024

PROJECT_ROOT = Path.cwd().parent

TAXI_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "taxi"
    / f"year={YEAR}"
)

WEATHER_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather"
    / f"year={YEAR}"
    / "station=USW00094728"
)

print(f"Taxi:    {TAXI_RAW_DIR}")
print(f"Weather: {WEATHER_RAW_DIR}")

Taxi:    c:\Users\derec\Desktop\Proyectos\BigData_project\data\raw\taxi\year=2024
Weather: c:\Users\derec\Desktop\Proyectos\BigData_project\data\raw\weather\year=2024\station=USW00094728


In [59]:
taxi_df = (
    spark.read
    .parquet(str(TAXI_RAW_DIR))
)

In [60]:
taxi_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [61]:
print(
    f"Número de columnas: "
    f"{len(taxi_df.columns)}"
)

Número de columnas: 19


## 3. Estrategia de procesamiento

Una prueba inicial de `cache()` sobre el DataFrame completo provocó la terminación del proceso Java de Spark en el entorno local. Por ello, el notebook adopta una estrategia más adecuada para este volumen:

- lectura directa desde Parquet;
- transformaciones *lazy*;
- construcción de múltiples métricas dentro de una sola agregación;
- materialización únicamente de resultados agregados pequeños.

De esta forma, Spark procesa las particiones necesarias sin intentar mantener los más de 41 millones de registros simultáneamente en memoria.


### 3.1 Variables auxiliares

Se crean columnas temporales para recuperar el archivo de origen, comparar el mes físico con el timestamp del viaje y calcular la duración. Estas columnas existen solo en el DataFrame de trabajo y no alteran la capa `raw`.


In [62]:
SOURCE_PATTERN = (
    r"yellow_tripdata_(\d{4})-(\d{2})\.parquet"
)

taxi_profile_df = (
    taxi_df

    # Archivo físico de origen
    .withColumn(
        "_source_file",
        F.input_file_name()
    )

    # Año indicado por el archivo
    .withColumn(
        "_source_year",
        F.regexp_extract(
            F.col("_source_file"),
            SOURCE_PATTERN,
            1
        ).cast("int")
    )

    # Mes indicado por el archivo
    .withColumn(
        "_source_month",
        F.regexp_extract(
            F.col("_source_file"),
            SOURCE_PATTERN,
            2
        ).cast("int")
    )

    # Año real según timestamp
    .withColumn(
        "_pickup_year",
        F.year(
            "tpep_pickup_datetime"
        )
    )

    # Mes real según timestamp
    .withColumn(
        "_pickup_month",
        F.month(
            "tpep_pickup_datetime"
        )
    )

    # Duración del viaje
    .withColumn(
        "_duration_minutes",
        (
            F.unix_timestamp(
                "tpep_dropoff_datetime"
            )
            -
            F.unix_timestamp(
                "tpep_pickup_datetime"
            )
        ) / 60.0
    )
)

### 3.2 Clasificación de variables

Las columnas se separan por tipo para aplicar controles apropiados. Las variables continuas no requieren el mismo tratamiento que las categóricas, temporales o identificadores geográficos.


In [63]:
NUMERIC_COLUMNS = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee"
]

In [64]:
CATEGORICAL_COLUMNS = [
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type"
]

In [65]:
LOCATION_COLUMNS = [
    "PULocationID",
    "DOLocationID"
]

In [66]:
TEMPORAL_COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
]

## 4. Construcción del profiling agregado

La agregación principal reúne en un solo plan de ejecución:

- número total de registros;
- nulos por columna;
- cardinalidad aproximada de variables categóricas;
- mínimo, media, desviación estándar y máximo de variables numéricas;
- rangos temporales y duración de los viajes;
- reglas de calidad sobre fechas, distancias, importes y pasajeros.

La cardinalidad se calcula de forma aproximada con `approx_count_distinct`, ya que para esta etapa interesa caracterizar las variables sin pagar el costo de un `distinct().count()` exacto sobre decenas de millones de filas.


### 4.1 Volumen y valores nulos


In [67]:
profile_expressions = [
    F.count(F.lit(1)).alias(
        "__total_rows"
    )
]

In [68]:
for column in taxi_df.columns:

    profile_expressions.append(
        F.sum(
            F.col(column)
            .isNull()
            .cast("long")
        ).alias(
            f"{column}__nulls"
        )
    )

### 4.2 Cardinalidad


In [69]:
CARDINALITY_COLUMNS = (
    CATEGORICAL_COLUMNS
    + LOCATION_COLUMNS
)

In [70]:
for column in CARDINALITY_COLUMNS:

    profile_expressions.append(
        F.approx_count_distinct(        
            F.col(column)
        ).alias(
            f"{column}__distinct"
        )
    )

### 4.3 Estadísticos numéricos


In [71]:
for column in NUMERIC_COLUMNS:

    profile_expressions.extend(
        [
            F.min(
                F.col(column)
            ).alias(
                f"{column}__min"
            ),

            F.avg(
                F.col(column)
            ).alias(
                f"{column}__mean"
            ),

            F.stddev(
                F.col(column)
            ).alias(
                f"{column}__std"
            ),

            F.max(
                F.col(column)
            ).alias(
                f"{column}__max"
            )
        ]
    )

### 4.4 Perfil temporal


In [72]:
profile_expressions.extend(
    [
        F.min(
            "tpep_pickup_datetime"
        ).alias(
            "__min_pickup"
        ),

        F.max(
            "tpep_pickup_datetime"
        ).alias(
            "__max_pickup"
        ),

        F.min(
            "tpep_dropoff_datetime"
        ).alias(
            "__min_dropoff"
        ),

        F.max(
            "tpep_dropoff_datetime"
        ).alias(
            "__max_dropoff"
        ),

        F.min(
            "_duration_minutes"
        ).alias(
            "__min_duration"
        ),

        F.avg(
            "_duration_minutes"
        ).alias(
            "__mean_duration"
        ),

        F.max(
            "_duration_minutes"
        ).alias(
            "__max_duration"
        )
    ]
)

### 4.5 Reglas de calidad

Las reglas siguientes identifican observaciones potencialmente problemáticas, pero **no implican eliminación automática**. Algunos valores —por ejemplo, importes negativos— pueden representar ajustes, reversos u otros casos legítimos y deberán evaluarse antes de limpiar.


In [73]:
profile_expressions.append(
    F.sum(
        F.when(
            F.col("_pickup_year") != YEAR,
            1
        ).otherwise(0)
    ).alias(
        "__pickup_outside_year"
    )
)

In [74]:
profile_expressions.append(
    F.sum(
        F.when(
            (
                F.col("_pickup_year")
                !=
                F.col("_source_year")
            )
            |
            (
                F.col("_pickup_month")
                !=
                F.col("_source_month")
            ),
            1
        ).otherwise(0)
    ).alias(
        "__source_date_mismatch"
    )
)

In [75]:
profile_expressions.append(
    F.sum(
        F.when(
            F.col("tpep_dropoff_datetime")
            <
            F.col("tpep_pickup_datetime"),
            1
        ).otherwise(0)
    ).alias(
        "__dropoff_before_pickup"
    )
)

In [76]:
profile_expressions.append(
    F.sum(
        F.when(
            F.col("_duration_minutes") <= 0,
            1
        ).otherwise(0)
    ).alias(
        "__non_positive_duration"
    )
)

In [77]:
## Distancia negativa
profile_expressions.append(
    F.sum(
        F.when(
            F.col("trip_distance") < 0,
            1
        ).otherwise(0)
    ).alias(
        "__negative_distance"
    )
)

In [78]:
## Distancia igual a cero
profile_expressions.append(
    F.sum(
        F.when(
            F.col("trip_distance") == 0,
            1
        ).otherwise(0)
    ).alias(
        "__zero_distance"
    )
)

In [79]:
for column in [
    "fare_amount",
    "tip_amount",
    "total_amount"
]:

    profile_expressions.append(
        F.sum(
            F.when(
                F.col(column) < 0,
                1
            ).otherwise(0)
        ).alias(
            f"__negative_{column}"
        )
    )

In [80]:
# Pasajeros igual a cero
profile_expressions.append(
    F.sum(
        F.when(
            F.col("passenger_count") == 0,
            1
        ).otherwise(0)
    ).alias(
        "__zero_passengers"
    )
)

In [81]:
## Pasajeros negativos
profile_expressions.append(
    F.sum(
        F.when(
            F.col("passenger_count") < 0,
            1
        ).otherwise(0)
    ).alias(
        "__negative_passengers"
    )
)

### 4.6 Ejecución

Todas las expresiones anteriores se ejecutan juntas. Spark recorre los archivos Parquet y devuelve una única fila agregada con las métricas del profiling.


In [82]:
profile_row = (
    taxi_profile_df
    .agg(
        *profile_expressions    # Desempaquetar la lista de expresiones
    )
    .first()
)

## 5. Resultados del profiling — NYC Yellow Taxi


### 5.1 Volumen

La base anual contiene **41,169,720 viajes** y **19 columnas**, un volumen suficiente para justificar el procesamiento distribuido con Spark incluso en un entorno local.


In [83]:
taxi_total_rows = (
    profile_row[
        "__total_rows"
    ]
)

taxi_total_columns = (
    len(taxi_df.columns)
)

print("=" * 60)
print("VOLUMEN DEL DATASET")
print("=" * 60)

print(
    f"Filas:    "
    f"{taxi_total_rows:,}"
)

print(
    f"Columnas: "
    f"{taxi_total_columns}"
)

VOLUMEN DEL DATASET
Filas:    41,169,720
Columnas: 19


### 5.2 Valores nulos

Cinco variables concentran exactamente el mismo número de valores ausentes: **4,091,232 registros (9.94 %)**. El resto de las columnas no presenta nulos.

La coincidencia exacta sugiere que no se trata de cinco problemas independientes, sino de un patrón estructural que conviene analizar como bloque.


In [84]:
null_profile_data = []

for column in taxi_df.columns:

    null_count = (
        profile_row[
            f"{column}__nulls"
        ]
    )

    null_pct = (
        null_count
        / taxi_total_rows
        * 100
    )

    null_profile_data.append(
        (
            column,
            int(null_count),
            float(null_pct)
        )
    )

In [85]:
null_profile_df = (
    spark.createDataFrame(
        null_profile_data,
        [
            "column",
            "null_count",
            "null_pct"
        ]
    )
)

In [86]:
null_profile_df = (
    null_profile_df
    .withColumn(
        "null_level",

        F.when(
            F.col("null_pct") == 0,
            "none"
        )

        .when(
            F.col("null_pct") < 1,
            "low"
        )

        .when(
            F.col("null_pct") < 10,
            "moderate"
        )

        .otherwise(
            "high"
        )
    )
)

In [87]:
null_profile_df.orderBy(
    F.desc("null_pct")
).show(
    n=50,
    truncate=False
)

+---------------------+----------+-----------------+----------+
|column               |null_count|null_pct         |null_level|
+---------------------+----------+-----------------+----------+
|RatecodeID           |4091232   |9.937478321445955|moderate  |
|congestion_surcharge |4091232   |9.937478321445955|moderate  |
|passenger_count      |4091232   |9.937478321445955|moderate  |
|Airport_fee          |4091232   |9.937478321445955|moderate  |
|store_and_fwd_flag   |4091232   |9.937478321445955|moderate  |
|improvement_surcharge|0         |0.0              |none      |
|mta_tax              |0         |0.0              |none      |
|total_amount         |0         |0.0              |none      |
|payment_type         |0         |0.0              |none      |
|tip_amount           |0         |0.0              |none      |
|PULocationID         |0         |0.0              |none      |
|fare_amount          |0         |0.0              |none      |
|trip_distance        |0         |0.0   

### 5.3 Patrón estructural de faltantes

Se comprueba si `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee` están ausentes simultáneamente y si ese mismo subconjunto coincide con `payment_type = 0`.


In [88]:
NULL_BLOCK_COLUMNS = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

all_null_condition = F.lit(True)

for column in NULL_BLOCK_COLUMNS:
    all_null_condition = (
        all_null_condition
        & F.col(column).isNull()
    )

any_null_condition = F.lit(False)

for column in NULL_BLOCK_COLUMNS:
    any_null_condition = (
        any_null_condition
        | F.col(column).isNull()
    )

In [89]:
missing_pattern_row = (
    taxi_df
    .agg(
        F.sum(
            all_null_condition.cast("long")
        ).alias("all_five_null"),

        F.sum(
            any_null_condition.cast("long")
        ).alias("any_of_five_null"),

        F.sum(
            (F.col("payment_type") == 0)
            .cast("long")
        ).alias("payment_type_zero"),

        F.sum(
            (
                all_null_condition
                & (F.col("payment_type") == 0)
            ).cast("long")
        ).alias("all_null_and_payment_zero")
    )
    .first()
)

print(
    f"Cinco variables simultáneamente NULL: "
    f"{missing_pattern_row['all_five_null']:,}"
)

print(
    f"Al menos una de las cinco NULL:       "
    f"{missing_pattern_row['any_of_five_null']:,}"
)

print(
    f"payment_type = 0:                     "
    f"{missing_pattern_row['payment_type_zero']:,}"
)

print(
    f"Ambas condiciones simultáneamente:    "
    f"{missing_pattern_row['all_null_and_payment_zero']:,}"
)

Cinco variables simultáneamente NULL: 4,091,232
Al menos una de las cinco NULL:       4,091,232
payment_type = 0:                     4,091,232
Ambas condiciones simultáneamente:    4,091,232


**Interpretación.** Las cuatro métricas toman el mismo valor: **4,091,232 registros**. Por tanto, las cinco variables son nulas exactamente en las mismas filas y ese bloque coincide también con `payment_type = 0`.

Esto indica una **estructura de registro distinta** dentro de la fuente, no un patrón aleatorio de missingness. En la etapa de limpieza no convendrá imputar estas columnas de manera independiente sin antes considerar el significado de este segmento.


### 5.4 Cardinalidad

Las variables categóricas presentan baja cardinalidad, mientras que las zonas de pickup y dropoff contienen alrededor de 260 identificadores distintos. Esto es consistente con su función como códigos discretos y facilita controles posteriores contra catálogos de referencia.


In [90]:
cardinality_data = []

for column in CARDINALITY_COLUMNS:

    approx_distinct = (
        profile_row[
            f"{column}__distinct"
        ]
    )

    cardinality_data.append(
        (
            column,
            int(approx_distinct)
        )
    )

In [91]:
cardinality_df = (
    spark.createDataFrame(
        cardinality_data,
        [
            "column",
            "approx_distinct"
        ]
    )
    .orderBy(
        "approx_distinct"
    )
)

cardinality_df.show(
    truncate=False
)

+------------------+---------------+
|column            |approx_distinct|
+------------------+---------------+
|store_and_fwd_flag|2              |
|VendorID          |4              |
|payment_type      |6              |
|RatecodeID        |7              |
|passenger_count   |10             |
|DOLocationID      |262            |
|PULocationID      |264            |
+------------------+---------------+



### 5.5 Perfil numérico

Los rangos revelan valores extremos importantes. En particular, `trip_distance`, `fare_amount` y `total_amount` presentan máximos que distorsionan fuertemente la media y la desviación estándar.

Por ejemplo, `trip_distance` tiene una media cercana a 5, pero una desviación estándar superior a 419 debido a observaciones extremas. Por ello, los estadísticos clásicos deben complementarse con percentiles robustos.


In [92]:
numeric_profile_data = []

for column in NUMERIC_COLUMNS:

    values = [
        profile_row[f"{column}__min"],
        profile_row[f"{column}__mean"],
        profile_row[f"{column}__std"],
        profile_row[f"{column}__max"]
    ]

    numeric_profile_data.append(
        (
            column,
            float(values[0]) if values[0] is not None else None,
            float(values[1]) if values[1] is not None else None,
            float(values[2]) if values[2] is not None else None,
            float(values[3]) if values[3] is not None else None
        )
    )

In [93]:
numeric_profile_df = (
    spark.createDataFrame(
        numeric_profile_data,
        [
            "column",
            "min",
            "mean",
            "std",
            "max"
        ]
    )
)

numeric_profile_df.show(
    n=50,
    truncate=False
)

+---------------------+--------+-------------------+-------------------+---------+
|column               |min     |mean               |std                |max      |
+---------------------+--------+-------------------+-------------------+---------+
|passenger_count      |0.0     |1.3339306338489314 |0.8158242156146573 |9.0      |
|trip_distance        |0.0     |4.976100785237054  |419.2304973495332  |398608.62|
|fare_amount          |-2261.2 |19.26850972995941  |76.71984026978807  |335544.44|
|extra                |-9.25   |1.3859530689059825 |1.8158781232411523 |65.99    |
|mta_tax              |-0.5    |0.4797774084934268 |0.13018302651337077|41.3     |
|tip_amount           |-300.0  |3.3078840186446565 |4.090523062522782  |999.99   |
|tolls_amount         |-140.63 |0.5615266120314077 |2.2405452034368993 |1702.88  |
|improvement_surcharge|-1.0    |0.9629933771713657 |0.2550554462404943 |2.0      |
|total_amount         |-2265.45|27.832813186929698 |78.05358621902298  |335550.94|
|con

### 5.6 Percentiles aproximados

Se calculan percentiles aproximados para caracterizar la distribución sin ordenar exactamente los 41 millones de registros.

Los percentiles centrales muestran valores plausibles para la mayor parte de los viajes, mientras que la cola extrema se separa drásticamente del resto. Esto confirma que los máximos no representan el comportamiento típico del dataset.

> **Nota:** el percentil 99.9 se calcula con `approxQuantile` y una tolerancia relativa de `0.001`. En una cola muy extrema, esa tolerancia puede hacer que el valor aproximado coincida con el máximo. Por ello, este percentil se utiliza como señal diagnóstica y no como cuantíl exacto.


In [94]:
QUANTILE_COLUMNS = [
    "trip_distance",
    "fare_amount",
    "total_amount",
    "_duration_minutes"
]

PROBABILITIES = [
    0.50,
    0.90,
    0.95,
    0.99,
    0.999
]

quantiles = (
    taxi_profile_df
    .approxQuantile(
        QUANTILE_COLUMNS,
        PROBABILITIES,
        0.001
    )
)

In [95]:
quantile_data = []

for column, values in zip(
    QUANTILE_COLUMNS,
    quantiles
):
    quantile_data.append(
        (
            column,
            *[float(value) for value in values]
        )
    )

quantile_df = spark.createDataFrame(
    quantile_data,
    [
        "column",
        "p50",
        "p90",
        "p95",
        "p99",
        "p999"
    ]
)

quantile_df.show(
    truncate=False
)

+-----------------+----+-----+------------------+------+-----------------+
|column           |p50 |p90  |p95               |p99   |p999             |
+-----------------+----+-----+------------------+------+-----------------+
|trip_distance    |1.76|8.8  |14.2              |19.95 |398608.62        |
|fare_amount      |13.5|40.8 |65.3              |80.0  |335544.44        |
|total_amount     |21.0|56.64|82.69             |104.88|335550.94        |
|_duration_minutes|13.0|33.0 |43.983333333333334|70.2  |9767.516666666666|
+-----------------+----+-----+------------------+------+-----------------+



### 5.7 Perfil temporal

El rango temporal incluye observaciones fuera de 2024 y existen duraciones negativas o extremadamente largas. La duración media (~17.5 minutos) es razonable como referencia global, pero los extremos requieren tratamiento posterior.


In [96]:
print("=" * 60)
print("PERFIL TEMPORAL")
print("=" * 60)

print(
    "Pickup mínimo:",
    profile_row["__min_pickup"]
)

print(
    "Pickup máximo:",
    profile_row["__max_pickup"]
)

print(
    "Dropoff mínimo:",
    profile_row["__min_dropoff"]
)

print(
    "Dropoff máximo:",
    profile_row["__max_dropoff"]
)

print()

print(
    "Duración mínima:",
    profile_row["__min_duration"]
)

print(
    "Duración media:",
    profile_row["__mean_duration"]
)

print(
    "Duración máxima:",
    profile_row["__max_duration"]
)

PERFIL TEMPORAL
Pickup mínimo: 2002-12-31 16:46:07
Pickup máximo: 2026-06-26 23:53:12
Dropoff mínimo: 2002-12-31 17:24:07
Dropoff máximo: 2026-06-27 20:59:10

Duración mínima: -1427.05
Duración media: 17.467908198874955
Duración máxima: 9767.516666666666


### 5.8 Controles de calidad

Los problemas más frecuentes son `trip_distance = 0`, importes negativos y `passenger_count = 0`. Las inconsistencias temporales estrictas son mucho menos frecuentes.

Es importante distinguir entre **anomalía estadística** y **registro inválido**: una distancia cero o un importe negativo no se eliminarán automáticamente hasta comprender su significado operativo.


In [97]:
quality_checks = [
    (
        "pickup_outside_year",
        profile_row[
            "__pickup_outside_year"
        ]
    ),

    (
        "source_date_mismatch",
        profile_row[
            "__source_date_mismatch"
        ]
    ),

    (
        "dropoff_before_pickup",
        profile_row[
            "__dropoff_before_pickup"
        ]
    ),

    (
        "non_positive_duration",
        profile_row[
            "__non_positive_duration"
        ]
    ),

    (
        "negative_distance",
        profile_row[
            "__negative_distance"
        ]
    ),

    (
        "zero_distance",
        profile_row[
            "__zero_distance"
        ]
    ),

    (
        "negative_fare_amount",
        profile_row[
            "__negative_fare_amount"
        ]
    ),

    (
        "negative_tip_amount",
        profile_row[
            "__negative_tip_amount"
        ]
    ),

    (
        "negative_total_amount",
        profile_row[
            "__negative_total_amount"
        ]
    ),

    (
        "zero_passengers",
        profile_row[
            "__zero_passengers"
        ]
    ),

    (
        "negative_passengers",
        profile_row[
            "__negative_passengers"
        ]
    )
]

In [98]:
# Añadimos porcentajes
quality_profile_data = [
    (
        check,
        int(count),
        float(
            count
            / taxi_total_rows
            * 100
        )
    )
    for check, count
    in quality_checks
]

In [99]:
quality_profile_df = (
    spark.createDataFrame(
        quality_profile_data,
        [
            "check",
            "count",
            "pct_rows"
        ]
    )
    .orderBy(
        F.desc("count")
    )
)

quality_profile_df.show(
    truncate=False
)

+---------------------+------+---------------------+
|check                |count |pct_rows             |
+---------------------+------+---------------------+
|zero_distance        |776305|1.885621276996783    |
|negative_fare_amount |731024|1.7756351026919786   |
|negative_total_amount|609344|1.4800780768001338   |
|zero_passengers      |401354|0.9748766812113369   |
|non_positive_duration|13510 |0.032815379847130365 |
|dropoff_before_pickup|1575  |0.0038256271842509494|
|negative_tip_amount  |1331  |0.0032329585918971517|
|source_date_mismatch |420   |0.0010201672491335866|
|pickup_outside_year  |56    |1.3602229988447822E-4|
|negative_passengers  |0     |0.0                  |
|negative_distance    |0     |0.0                  |
+---------------------+------+---------------------+



### 5.9 Distribución de variables categóricas

Las distribuciones permiten identificar categorías dominantes y valores poco frecuentes sin asumir todavía que estos últimos sean inválidos.

Destacan:

- `VendorID = 2` concentra alrededor del 76 % de los viajes;
- `passenger_count = 1` representa cerca del 70 %;
- `RatecodeID = 1` domina con más del 84 %;
- `store_and_fwd_flag = N` representa casi el 90 %;
- `payment_type = 1` es la categoría más frecuente.

Los códigos raros (`VendorID` 6/7, `RatecodeID` 99, etc.) deberán contrastarse posteriormente con documentación o catálogos antes de clasificarlos como errores.


In [100]:
LOW_CARDINALITY_COLUMNS = [
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type"
]

In [101]:
for column in LOW_CARDINALITY_COLUMNS:

    print("=" * 60)
    print(column)
    print("=" * 60)

    (
        taxi_df
        .groupBy(column)
        .count()
        .withColumn(
            "pct",
            F.round(
                F.col("count")
                / taxi_total_rows
                * 100,
                4
            )
        )
        .orderBy(
            F.desc("count")
        )
        .show(
            n=50,
            truncate=False
        )
    )

VendorID
+--------+--------+-------+
|VendorID|count   |pct    |
+--------+--------+-------+
|2       |31451503|76.3947|
|1       |9715918 |23.5997|
|6       |2069    |0.005  |
|7       |230     |6.0E-4 |
+--------+--------+-------+

passenger_count
+---------------+--------+-------+
|passenger_count|count   |pct    |
+---------------+--------+-------+
|1              |28632703|69.548 |
|2              |5410774 |13.1426|
|NULL           |4091232 |9.9375 |
|3              |1282073 |3.1141 |
|4              |814889  |1.9793 |
|0              |401354  |0.9749 |
|5              |320613  |0.7788 |
|6              |215798  |0.5242 |
|8              |192     |5.0E-4 |
|7              |56      |1.0E-4 |
|9              |36      |1.0E-4 |
+---------------+--------+-------+

RatecodeID
+----------+--------+-------+
|RatecodeID|count   |pct    |
+----------+--------+-------+
|1         |34651027|84.1663|
|NULL      |4091232 |9.9375 |
|2         |1406870 |3.4172 |
|99        |466974  |1.1343 |
|5 

## 6. Perfil meteorológico — NOAA

La base meteorológica es mucho menor que la de Taxi, por lo que puede perfilarse directamente. Se expanden las observaciones del JSON y se valida la cobertura temporal por tipo de variable.


In [102]:
weather_raw_df = (
    spark.read
    .option(
        "multiLine",
        True
    )
    .json(
        str(WEATHER_RAW_DIR)
    )
)

In [103]:
weather_df = (
    weather_raw_df
    .select(
        F.explode(      
            "results"
        ).alias(
            "record"
        )
    )
    .select(
        "record.*"
    )
    .withColumn(
        "observation_date",
        F.to_date("date")
    )
)

### 6.1 Cobertura general


In [104]:
weather_total_rows = (
    weather_df.count()
)

weather_distinct_days = (
    weather_df
    .select(
        "observation_date"
    )
    .distinct()
    .count()
)

print(
    f"Observaciones: "
    f"{weather_total_rows:,}"
)

print(
    f"Días distintos: "
    f"{weather_distinct_days}"
)

Observaciones: 1,826
Días distintos: 366


### 6.2 Completitud por variable


In [105]:
weather_completeness_df = (
    weather_df
    .groupBy(
        "datatype"
    )
    .agg(
        F.count("*").alias(
            "observations"
        ),

        F.countDistinct(
            "observation_date"
        ).alias(
            "distinct_days"
        )
    )
    .withColumn(
        "expected_days",
        F.lit(366)
    )
    .withColumn(
        "missing_days",
        F.col("expected_days")
        -
        F.col("distinct_days")
    )
    .withColumn(
        "completeness_pct",
        F.round(
            F.col("distinct_days")
            /
            F.col("expected_days")
            * 100,
            2
        )
    )
    .orderBy(
        "datatype"
    )
)

weather_completeness_df.show(
    truncate=False
)

+--------+------------+-------------+-------------+------------+----------------+
|datatype|observations|distinct_days|expected_days|missing_days|completeness_pct|
+--------+------------+-------------+-------------+------------+----------------+
|AWND    |362         |362          |366          |4           |98.91           |
|PRCP    |366         |366          |366          |0           |100.0           |
|SNOW    |366         |366          |366          |0           |100.0           |
|TMAX    |366         |366          |366          |0           |100.0           |
|TMIN    |366         |366          |366          |0           |100.0           |
+--------+------------+-------------+-------------+------------+----------------+



**Interpretación.** NOAA cubre los **366 días de 2024**. `PRCP`, `SNOW`, `TMAX` y `TMIN` están completas, mientras que `AWND` dispone de 362 observaciones, equivalente a una completitud de **98.91 %** para esa variable y **99.78 %** para el conjunto meteorológico total.

Las cuatro ausencias de `AWND` ya fueron identificadas durante la ingesta y se conservarán sin imputar hasta la etapa de transformación.


## 7. Resumen de hallazgos

| Fuente | Hallazgo | Magnitud / evidencia | Implicación |
|---|---|---:|---|
| NYC Taxi | Bloque estructural de nulos | 4,091,232 filas (9.94 %) | Analizar como segmento; no imputar variables por separado |
| NYC Taxi | `payment_type = 0` coincide con el bloque nulo | 4,091,232 filas | Refuerza que existe una estructura de registro diferenciada |
| NYC Taxi | Distancia igual a cero | 776,305 filas (1.89 %) | Requiere interpretación antes de filtrar |
| NYC Taxi | Importes negativos | 731,024 `fare_amount`; 609,344 `total_amount` | Pueden incluir ajustes o reversos |
| NYC Taxi | Duración no positiva | 13,510 filas | Inconsistencia temporal a revisar |
| NYC Taxi | Dropoff anterior al pickup | 1,575 filas | Inconsistencia inequívoca |
| NYC Taxi | Mes del timestamp no coincide con el archivo | 420 filas | Problema de consistencia fuente-fecha |
| NYC Taxi | Pickup fuera de 2024 | 56 filas | Registros fuera del período objetivo |
| NYC Taxi | Valores extremos | máximos muy alejados de P50–P99 | Usar reglas robustas / percentiles en limpieza |
| NOAA | `AWND` incompleto | 362/366 días | Evaluar imputación o conservación de nulos |
| NOAA | Resto de variables | 366/366 días | Cobertura completa |


## 8. Conclusiones

El profiling permitió caracterizar la calidad de aproximadamente **41.17 millones de viajes de NYC Yellow Taxi** y de la fuente meteorológica NOAA antes de realizar cualquier transformación.

En Taxi se identificaron tres grupos principales de problemas:

1. **Inconsistencias temporales**, incluyendo registros fuera de 2024, diferencias entre el mes del archivo y el timestamp del viaje, y viajes con duración no positiva.
2. **Valores numéricos extremos**, especialmente en distancia, duración e importes monetarios. Los máximos son suficientemente altos para distorsionar la media y la desviación estándar, por lo que los percentiles ofrecen una representación más robusta del comportamiento típico.
3. **Patrones estructurados de valores faltantes**. Cinco variables presentan exactamente 4,091,232 valores nulos y coinciden con `payment_type = 0`, evidenciando un subconjunto específico de registros que deberá tratarse como tal.

NOAA presenta una calidad considerablemente mayor: cubre los 366 días de 2024 y alcanza una completitud global aproximada de **99.78 %**, con únicamente cuatro observaciones ausentes en `AWND`.

Ningún registro fue eliminado ni modificado durante esta etapa. Los hallazgos obtenidos servirán como base para definir reglas explícitas, reproducibles y justificadas de limpieza y transformación en la siguiente fase del pipeline.


In [106]:
# Desactivamos la sesión de Spark al final del notebook
# spark.stop()